# Лекция 10 · Анализ ошибок и различия dev/test / Error analysis and distribution mismatch

**RU.** Реальные рукописные цифры UCI Optical Recognition of Handwritten Digits. Один объект — изображение 8×8, признаки — 64 счётчика пикселей 0…16, цель — цифра 0…9. Исходный официальный train содержит 3823 изображения от 30 авторов, test — 1797 изображений от других 13 авторов. В этой учебной задаче официальный train называется source, официальный test — target. Source делится на train/train-dev (80/20), target — на dev/test (50/50), seed=1010, stratify по цифре. Мы намеренно используем половину официального test для разработки и поэтому НЕ заявляем результат на стандартном benchmark. Разбиение внутри target не гарантирует новых авторов между dev/test: идентификаторов авторов в этих CSV нет. Все варианты модели получают одинаковые индексы. Test не участвует в выборе модели.

**EN.** Real UCI Optical Recognition of Handwritten Digits. Each observation is an 8×8 image, features are 64 pixel counts in 0…16, and the target is a digit 0…9. The official training file contains 3823 images from 30 writers; the official test file has 1797 images from 13 different writers. For this teaching protocol we call them source and target. Split source into train/train-dev (80/20), target into dev/test (50/50), seed=1010, stratified by digit. We deliberately repurpose half of the official test for development: this is NOT a standard benchmark score. Target dev/test are not guaranteed writer-disjoint because the CSV files have no writer IDs. All candidates use identical indices. Test is excluded from selection.

**Источник / Source:** [UCI, Alpaydin & Kaynak, Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), DOI 10.24432/C50P49, CC BY 4.0 (see source terms). Cite the dataset when reusing. CPU, NumPy, pandas, scikit-learn, matplotlib; the official ZIP is downloaded once and SHA-256 checked. No synthetic observations or labels. Historical sources: Archive/Tasks/Task4_ErrorEstimation.ipynb and Task5_Invariant.ipynb. Updated 2026: source/target protocol, explicit error tables, honest tests, no Google writes from notebooks.


In [ ]:
import json, hashlib, zipfile, urllib.request  # RU: загружаем и проверяем архив; EN: download and verify the archive.
from pathlib import Path  # RU: задаём путь к кэшу; EN: define the cache path.
import numpy as np  # RU: работаем с пикселями и индексами; EN: manipulate pixels and indices.
import pandas as pd  # RU: строим таблицы ошибок; EN: construct error tables.
import matplotlib.pyplot as plt  # RU: показываем реальные изображения; EN: display real images.
from sklearn.model_selection import train_test_split  # RU: разделяем реальные наблюдения; EN: split real observations.
from sklearn.svm import SVC  # RU: обучаем компактный классификатор; EN: train a compact classifier.
from sklearn.metrics import confusion_matrix  # RU: независимо проверяем таблицу ошибок; EN: independently check confusion counts.
cache = Path("optdigits.zip")  # RU: повторно используем локальный архив; EN: reuse a local archive.
if not cache.exists():  # RU: скачиваем только при отсутствии; EN: download only when absent.
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/80/optical+recognition+of+handwritten+digits.zip", cache)  # RU: используем официальный UCI; EN: use the official UCI source.
assert hashlib.sha256(cache.read_bytes()).hexdigest() == "0d7b054fea010270e9b3f06411c654c5e59547732ad626381980baffe0a23fb0"  # RU: проверяем неизменность данных; EN: verify data integrity.
with zipfile.ZipFile(cache) as archive:  # RU: открываем CSV внутри ZIP без распаковки; EN: open CSV members without extraction.
    source = np.loadtxt(archive.open("optdigits.tra"), delimiter=",")  # RU: читаем исходных авторов; EN: read source writers.
    target = np.loadtxt(archive.open("optdigits.tes"), delimiter=",")  # RU: читаем других авторов; EN: read different target writers.
xs, ys = source[:, :64] / 16.0, source[:, 64].astype(int)  # RU: нормируем по известному максимуму; EN: normalize by the known maximum.
xt, yt = target[:, :64] / 16.0, target[:, 64].astype(int)  # RU: применяем то же преобразование к target; EN: apply the same transform to target.
train, train_dev = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=1010)  # RU: оставляем source-контроль; EN: hold out source validation.
dev, test = train_test_split(np.arange(len(yt)), test_size=0.5, stratify=yt, random_state=1010)  # RU: отделяем целевой test; EN: reserve target test.
assert not set(train) & set(train_dev) and not set(dev) & set(test)  # RU: запрещаем пересечение внутри источника; EN: disallow overlap within a source.
assert len(source) == 3823 and len(target) == 1797  # RU: проверяем полноту файлов; EN: check file completeness.
baseline = SVC(C=0.1, kernel="rbf", gamma="scale").fit(xs[train], ys[train])  # RU: fit видит только train; EN: fit sees training data only.
dev_pred = baseline.predict(xt[dev])  # RU: анализируем dev до выбора следующей модели; EN: inspect dev before selecting the next model.
print("train / train-dev / dev / test:", len(train), len(train_dev), len(dev), len(test))  # RU: выводим размеры разбиений; EN: display partition sizes.


## 1 · Срезы ошибок / Error slices

**RU · 5–10 минут.** Напишите `error_table(y_true, y_pred)`: DataFrame с колонками digit, n, errors, error_rate и ровно 10 строками в порядке 0…9. Для пустого класса n=0, errors=0, error_rate=NaN. Доля ошибок — errors/n, не доля среди всех ошибок. Вызовите на dev, найдите худший класс и сообщите знаменатель. Не меняйте модель по test.
**EN · 5–10 minutes.** Implement `error_table(y_true, y_pred)`: a DataFrame with digit, n, errors, error_rate, exactly ten ordered rows for 0…9. An absent class has n=0, errors=0, rate=NaN. Rate means errors/n, not share of all errors. Run on dev, identify the worst class and report its denominator. Do not tune on test.

In [ ]:
print("Overall dev error:", np.mean(dev_pred != yt[dev]))  # RU: показываем агрегат перед детализацией; EN: show the aggregate before slicing.

In [ ]:
# RU: Ваше решение задания 1. EN: Your solution for exercise 1.
def error_table(y_true, y_pred):  # RU: считаем ошибки по истинному классу; EN: count errors by true class.
    raise NotImplementedError("TODO / Ваш код")  # RU: замените заглушку; EN: replace this placeholder.

In [ ]:
table = error_table(yt[dev], dev_pred)  # RU: получаем срезы реальных данных; EN: obtain real-data slices.
cm = confusion_matrix(yt[dev], dev_pred, labels=range(10))  # RU: строим независимый эталон; EN: construct an independent reference.
assert list(table.columns) == ["digit", "n", "errors", "error_rate"]  # RU: проверяем интерфейс; EN: check the interface.
assert np.array_equal(table["n"], cm.sum(axis=1))  # RU: сверяем размеры срезов; EN: verify slice sizes.
assert np.array_equal(table["errors"], cm.sum(axis=1) - np.diag(cm))  # RU: сверяем ошибки вне диагонали; EN: verify off-diagonal errors.
assert np.allclose(table["error_rate"], table["errors"] / table["n"])  # RU: проверяем правильный знаменатель; EN: check the denominator.
empty = error_table(yt[dev][yt[dev] != 9], dev_pred[yt[dev] != 9])  # RU: исключаем реальный класс для проверки края; EN: remove a real class to test an edge case.
assert np.isnan(empty.loc[9, "error_rate"])  # RU: не превращаем отсутствие данных в нулевую ошибку; EN: do not equate missing data with zero error.
display(table)  # RU: показываем рассчитанные срезы; EN: display calculated slices.

## 2 · Журнал и изображения / Audit log and images

**RU · 5–10 минут.** Реализуйте `audit_errors(source_ids, y_true, y_pred)`: только неверные ответы, колонки source_id/truth/prediction, сортировка по source_id, новый индекс 0…N−1. Передайте `dev` как идентификаторы строк файла optdigits.tes. Покажите первые восемь ошибок из журнала, подпишите истинную и предсказанную цифру. Для двух примеров запишите гипотезу и способ её проверить; нельзя объявлять метку ошибочной только потому, что модель не согласна.
**EN · 5–10 minutes.** Implement `audit_errors(source_ids, y_true, y_pred)`: errors only, source_id/truth/prediction columns, ascending source_id, fresh 0…N−1 index. Pass `dev` as optdigits.tes row IDs. Display the first eight errors with true and predicted digits. For two examples state a hypothesis and how to test it; disagreement with a model does not establish a wrong label.

In [ ]:
print("Target row ID:", int(dev[0]), "true label:", int(yt[dev[0]]))  # RU: различаем позицию и исходный индекс; EN: distinguish position from source index.

In [ ]:
# RU: Ваше решение задания 2. EN: Your solution for exercise 2.
def audit_errors(source_ids, y_true, y_pred):  # RU: сохраняем связь ошибки с изображением; EN: retain the image identity for each error.
    raise NotImplementedError("TODO / Ваш код")  # RU: замените заглушку; EN: replace this placeholder.

In [ ]:
audit = audit_errors(dev, yt[dev], dev_pred)  # RU: строим воспроизводимый журнал; EN: build a reproducible audit log.
expected = sorted(dev[dev_pred != yt[dev]].tolist())  # RU: независимо выбираем индексы ошибок; EN: independently select error IDs.
assert audit["source_id"].tolist() == expected  # RU: проверяем состав и порядок; EN: check membership and order.
assert np.array_equal(audit["truth"], yt[audit["source_id"].to_numpy()])  # RU: сверяем метки с оригиналом; EN: verify labels against the source.
assert (audit["truth"] != audit["prediction"]).all()  # RU: исключаем правильные ответы; EN: exclude correct predictions.
fig, axes = plt.subplots(2, 4, figsize=(10, 5))  # RU: готовим сетку реальных ошибок; EN: prepare a real-error grid.
for ax in axes.flat:  # RU: очищаем все оси, включая пустые; EN: clear every axis including empty ones.
    ax.axis("off")  # RU: скрываем координаты пикселей; EN: hide pixel coordinates.
for ax, record in zip(axes.flat, audit.head(8).itertuples()):  # RU: используем фиксированный порядок; EN: use the fixed ordering.
    ax.imshow(xt[record.source_id].reshape(8, 8), cmap="gray_r")  # RU: берём исходное изображение; EN: retrieve the source image.
    ax.set_title(f"{record.source_id}: {record.truth} → {record.prediction}")  # RU: подписываем ошибку; EN: label the error.
plt.tight_layout(); plt.show()  # RU: показываем читаемую сетку; EN: show a readable grid.

## 3 · Разрывы ошибок / Error gaps

**RU · 5–10 минут.** Реализуйте `diagnostic_gaps(train_error, train_dev_error, dev_error)` с двумя ключами generalization_gap=train_dev−train и mismatch_gap=dev−train_dev. Сохраняйте отрицательные значения: это диагностика, не штраф. Все входы — доли 0…1. Посчитайте значения baseline и объясните, почему mismatch_gap сам по себе не доказывает причину ухудшения. Train-dev не используется для fit.
**EN · 5–10 minutes.** Implement two signed differences: generalization_gap=train_dev−train and mismatch_gap=dev−train_dev. Preserve negative values; these are diagnostics, not penalties. Inputs are fractions in 0…1. Compute the baseline gaps and explain why a mismatch gap alone does not prove the cause of degradation. Never fit on train-dev.

In [ ]:
print("Train-dev source size:", len(train_dev), "Target dev size:", len(dev))  # RU: проверяем различие источников; EN: check the distinct sources.

In [ ]:
# RU: Ваше решение задания 3. EN: Your solution for exercise 3.
def diagnostic_gaps(train_error, train_dev_error, dev_error):  # RU: сравниваем три ошибки; EN: compare three error rates.
    raise NotImplementedError("TODO / Ваш код")  # RU: замените заглушку; EN: replace this placeholder.

In [ ]:
et = float(np.mean(baseline.predict(xs[train]) != ys[train]))  # RU: измеряем ошибку на fit-данных; EN: measure training error.
es = float(np.mean(baseline.predict(xs[train_dev]) != ys[train_dev]))  # RU: измеряем source-контроль; EN: measure source holdout error.
ed = float(np.mean(dev_pred != yt[dev]))  # RU: измеряем target dev; EN: measure target development error.
gaps = diagnostic_gaps(et, es, ed)  # RU: считаем диагностические разрывы; EN: calculate diagnostic gaps.
assert np.isclose(gaps["generalization_gap"], es-et)  # RU: сверяем первое вычитание; EN: verify the first difference.
assert np.isclose(gaps["mismatch_gap"], ed-es)  # RU: сверяем второе вычитание; EN: verify the second difference.
assert np.isclose(diagnostic_gaps(ed, es, et)["mismatch_gap"], et-es)  # RU: проверяем сохранение знака; EN: check signed differences.
print(gaps)  # RU: выводим результат для интерпретации; EN: display results for interpretation.

## 4 · Выбор следующего эксперимента / Select the next experiment

**RU · 5–10 минут.** Реализуйте `select_candidate(records)`, возвращающую C из строки с минимальным dev_error; при равенстве — меньший C. Разрешены только train_error/train_dev_error/dev_error и C, никаких test-метрик. Обучите заранее объявленные C=[0.01,0.1,1,10] на одном train, заполните журнал, выберите C, зафиксируйте его и один раз измерьте test. Не меняйте кандидатов после просмотра test. Напишите вывод из трёх предложений: наблюдение, гипотеза, следующий эксперимент.
**EN · 5–10 minutes.** Return C from the row with lowest dev_error, using smaller C for ties. Only training/source-holdout/development errors are allowed; no test metrics. Train the predeclared C=[0.01,0.1,1,10] on the same train, fill the journal, select and freeze C, then evaluate test once. Do not change candidates after opening test. Write three sentences: observation, hypothesis, next experiment.

In [ ]:
print("Predeclared candidates:", [0.01, 0.1, 1, 10])  # RU: задаём бюджет до оценки; EN: declare the budget before evaluation.

In [ ]:
# RU: Ваше решение задания 4. EN: Your solution for exercise 4.
def select_candidate(records):  # RU: выбираем кандидата без test; EN: select without test data.
    raise NotImplementedError("TODO / Ваш код")  # RU: замените заглушку; EN: replace this placeholder.

In [ ]:
records, fitted = [], {}  # RU: храним измерения и обученные модели; EN: store measurements and fitted models.
for c in [0.01, 0.1, 1, 10]:  # RU: перебираем фиксированный бюджет; EN: iterate over the fixed budget.
    model = SVC(C=c, kernel="rbf", gamma="scale").fit(xs[train], ys[train])  # RU: обучаем только на train; EN: fit on training data only.
    fitted[c] = model  # RU: сохраняем модель без переобучения на test; EN: retain the model without test refitting.
    records.append({"C": c, "train_error": float(np.mean(model.predict(xs[train]) != ys[train])), "train_dev_error": float(np.mean(model.predict(xs[train_dev]) != ys[train_dev])), "dev_error": float(np.mean(model.predict(xt[dev]) != yt[dev]))})  # RU: собираем сопоставимые ошибки; EN: collect comparable errors.
chosen_c = select_candidate(records)  # RU: фиксируем победителя по dev; EN: freeze the development winner.
assert chosen_c == sorted(records, key=lambda r: (r["dev_error"], r["C"]))[0]["C"]  # RU: сверяем правило ранжирования; EN: verify the ranking rule.
ties = [{**r, "dev_error": records[0]["dev_error"]} for r in records]  # RU: проверяем равенство на реальных записях; EN: test ties using actual candidate records.
assert select_candidate(ties) == min(r["C"] for r in records)  # RU: сверяем правило ничьей; EN: verify tie breaking.
final_test_error = float(np.mean(fitted[chosen_c].predict(xt[test]) != yt[test]))  # RU: открываем test после фиксации; EN: open test after freezing selection.
display(pd.DataFrame(records)); print("Frozen C / test error:", chosen_c, final_test_error)  # RU: отделяем выбор от финальной оценки; EN: separate selection and final evaluation.